# NovelReader — WebUI Launcher

Chỉ khởi động FastAPI và zrok, rồi in đường dẫn, mã truy cập, log. Dịch, TTS và Hugging Face được điều khiển trên WebUI.


In [ ]:
# Notebook Colab — chạy WebUI, lấy link + mã truy cập + logcat.
# Chỉ điền token thật trong bản notebook cá nhân, không commit vào public GitHub.
HF_TOKEN_INLINE = ""
ZROK_ACCOUNT_TOKEN_INLINE = ""

import os, sys, subprocess, urllib.request, zipfile
from io import BytesIO
from pathlib import Path

if HF_TOKEN_INLINE.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN_INLINE.strip()
if ZROK_ACCOUNT_TOKEN_INLINE.strip():
    os.environ["ZROK_ACCOUNT_TOKEN"] = ZROK_ACCOUNT_TOKEN_INLINE.strip()

base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repo_dir = base / "NovelReader_T4_Web"
if not (repo_dir / "notebook/webui_launcher.py").is_file():
    if (repo_dir / "main.py").is_file():
        raise RuntimeError("Code repo cũ. Cập nhật checkout sau khi job dừng để giữ dữ liệu.")
    clone = subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/dungbeommm/NovelReader_Translate.git", str(repo_dir)],
        text=True, capture_output=True, timeout=120)
    if clone.returncode:
        print("git clone thất bại, thử ZIP...", flush=True)
        with urllib.request.urlopen(
            "https://github.com/dungbeommm/NovelReader_Translate/archive/refs/heads/main.zip", timeout=120
        ) as response:
            data = response.read()
        repo_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(BytesIO(data)) as archive:
            for member in archive.infolist():
                parts = Path(member.filename).parts[1:]
                if not parts:
                    continue
                dest = repo_dir.joinpath(*parts)
                if not dest.resolve().is_relative_to(repo_dir.resolve()):
                    raise ValueError("ZIP chứa đường dẫn không an toàn.")
                if member.is_dir():
                    dest.mkdir(parents=True, exist_ok=True)
                else:
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    dest.write_bytes(archive.read(member))

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
os.chdir(repo_dir)
from notebook.webui_launcher import launch
session = launch(repo_dir)


In [ ]:
# Chạy lại cell này để xem logcat mới nhất; các thao tác dịch/audio thực hiện trên WebUI.
from notebook.webui_launcher import print_result, print_logs
print_result(session)
print_logs(session, lines=70)
